# Spectral analysis

This notebook is for computing the spectral analysis as described in the manuscript.

In [ ]:
import xarray as xr 
import matplotlib.pyplot as plt 
import matplotlib.colors as mcolors
from matplotlib.ticker import LogFormatter,LogLocator

import numpy as np 
import pandas as pd

import xskillscore

import xrft

from pyinterp import fill, Axis, TemporalAxis, Grid3D, Grid2D
n_workers = 10

import matplotlib.colors as mcolors

import colorcet as cc 
import cmocean 

from mapping.src.functions_interp import fill_nan

In [ ]:
path_data = "../.." # Change path according to where data are saved

## Importing data

QG reconstruction 

In [ ]:
ds_QG = xr.open_mfdataset(f"{path_data}/data/mapping_outputs/config_QG/*.nc")
ssh_bm_QG = ds_QG.ssh_bm.load()
ssh_bm_QG = ssh_bm_QG.rename({"lon":"longitude","lat":"latitude"})

QG/SW reconstruction 

In [ ]:
ds_QGSW = xr.open_mfdataset(f"{path_data}/data/mapping_outputs/config_QGSW/*.nc")
# - BALANCED MOTIONS BM - #
ssh_bm_QGSW = ds_QGSW.ssh_bm.load()
ssh_bm_QGSW = ssh_bm_QGSW.rename({"lon":"longitude","lat":"latitude"})
# - INTERNAL TIDE IT - #
ssh_it_QGSW = ds_QGSW.ssh_it.load()
ssh_it_QGSW = ssh_it_QGSW.rename({"lon":"longitude","lat":"latitude"})

QG/SW reconstruction no time 

In [ ]:
ds_QGSW_notime = xr.open_mfdataset(f"{path_data}/data/mapping_outputs/config_QGSW_notime/*.nc")
# - BALANCED MOTIONS BM - #
ssh_bm_QGSW_notime = ds_QGSW_notime.ssh_bm.load()
ssh_bm_QGSW_notime = ssh_bm_QGSW_notime.rename({"lon":"longitude","lat":"latitude"})
# - INTERNAL TIDE IT - #
ssh_it_QGSW_notime = ds_QGSW_notime.ssh_it.load()
ssh_it_QGSW_notime = ssh_it_QGSW_notime.rename({"lon":"longitude","lat":"latitude"})

Reference dataset 

In [ ]:
ds_truth = xr.open_mfdataset(f"{path_data}/data/OSSE/ref/*.nc")
ds_truth = ds_truth.sel(longitude=slice(ssh_bm_QG.longitude[0],ssh_bm_QG.longitude[-1]),
                        latitude=slice(ssh_bm_QG.latitude[0],ssh_bm_QG.latitude[-1]),
                        time = slice(ssh_bm_QG.time[0],ssh_bm_QG.time[-1]),drop=True)
ssh_bm_truth = ds_truth.ssh_bm.load()
ssh_it_12h_truth = ds_truth.ssh_it_12h.load()
ssh_it_truth = ds_truth.ssh_it.load()
ssh_tot_truth = ds_truth.ssh.load()

In [ ]:
ssh_bm_truth = fill_nan(ssh_bm_truth)
ssh_it_truth = fill_nan(ssh_it_truth)
ssh_tot_truth = fill_nan(ssh_tot_truth)

## Computing 2D spectrum

In [ ]:
def _psd(ssh):
    ssh_copy = ssh.copy()
    ssh_copy['time'] = (ssh_copy.time - ssh_copy.time[0]) / np.timedelta64(1, 'D')

    psd = xrft.power_spectrum(
        ssh_copy, 
        dim=None, 
        detrend="constant", 
        window=True).compute()

    # Isotropize
    psd = xrft.isotropize(psd,fftdim=[f'freq_longitude',f'freq_latitude'])

    psd = psd.where(psd.freq_time>0,drop=True)

    return psd 

In [ ]:
### COMPUTING PSDs RECONSTRUCTION ###
# - Balanced Motions - # 
# psd_bm_QG = _psd(ssh_bm_QG)
psd_bm_QGSW = _psd(ssh_bm_QGSW)
# - Internal Tide - # 
psd_it_QGSW = _psd(ssh_it_QGSW)
# - Total - # 
psd_tot_QGSW = _psd(ssh_bm_QGSW+ssh_it_QGSW)
### COMPUTING PSDs REFERENCE ###
# - Balanced Motions - # 
psd_bm_truth = _psd(ssh_bm_truth)
# - Internal Tide - # 
psd_it_truth = _psd(ssh_it_truth)
# - Total - # 
psd_tot_truth = _psd(ssh_tot_truth)

In [ ]:
### REMOVING FIRST WAVELENGTH ### 
# - Balanced Motions - # 
# psd_bm_QG = psd_bm_QG.where(psd_bm_QG.freq_r>0.4,drop=True)
psd_bm_QGSW = psd_bm_QGSW.where(psd_bm_QGSW.freq_r>0.4,drop=True)
# - Internal Tide - # 
psd_it_QGSW = psd_it_QGSW.where(psd_it_QGSW.freq_r>0.4,drop=True)
# - Total - # 
psd_tot_QGSW = psd_tot_QGSW.where(psd_tot_QGSW.freq_r>0.4,drop=True)
### COMPUTING PSDs REFERENCE ###
# - Balanced Motions - # 
psd_bm_truth = psd_bm_truth.where(psd_bm_truth.freq_r>0.4,drop=True)
# - Internal Tide - # 
psd_it_truth = psd_it_truth.where(psd_it_truth.freq_r>0.4,drop=True)
# - Total - # 
psd_tot_truth = psd_tot_truth.where(psd_tot_truth.freq_r>0.4,drop=True)

In [ ]:
psd_err_tot = _psd(ssh_bm_QGSW+ssh_it_QGSW-ssh_tot_truth)
psd_err_bm = _psd(ssh_bm_QGSW-ssh_bm_truth)
psd_err_it = _psd(ssh_it_QGSW-ssh_it_truth)

## Plot Figure **03**

In [ ]:
# cmap = "magma"
cmap=cc.cm.CET_L17
# cmap_score = "afmhot"
cmap_score = cc.cm.CET_L18

fig = plt.figure(figsize=(12, 12), dpi=300)

# ----------------------------
# Outer grid: 1 row, 2 sections
# Left section: first two columns of plots
# Right section: third column of plots
outer_gs = fig.add_gridspec(1, 2, width_ratios=[2, 0.93], wspace=0.3)

# Left section: 3x2 subgrid with its own spacing
left_gs = outer_gs[0].subgridspec(3, 2, wspace=0.2, hspace=0.25)

ax00 = fig.add_subplot(left_gs[0, 0])
ax01 = fig.add_subplot(left_gs[0, 1])
ax10 = fig.add_subplot(left_gs[1, 0])
ax11 = fig.add_subplot(left_gs[1, 1])
ax20 = fig.add_subplot(left_gs[2, 0])
ax21 = fig.add_subplot(left_gs[2, 1])

# Right section: 3x1 subgrid with its own spacing
right_gs = outer_gs[1].subgridspec(3, 1, hspace=0.25)

ax02 = fig.add_subplot(right_gs[0, 0])
ax12 = fig.add_subplot(right_gs[1, 0])
ax22 = fig.add_subplot(right_gs[2, 0])

# ----------------------------
# Example pcolormesh plots
norm_main = mcolors.LogNorm(vmin=1e-10, vmax=1e-4)
norm_right = mcolors.LogNorm(vmin=1e-10, vmax=1e-3)

# First two columns
plots_main = []
plots_main.append(ax00.pcolormesh(111/psd_tot_truth.freq_r, 1/psd_tot_truth.freq_time, psd_tot_truth, norm=norm_main, cmap=cmap))
plots_main.append(ax01.pcolormesh(111/psd_tot_QGSW.freq_r, 1/psd_tot_QGSW.freq_time, psd_tot_QGSW, norm=norm_main, cmap=cmap))
plots_main.append(ax10.pcolormesh(111/psd_bm_truth.freq_r, 1/psd_bm_truth.freq_time, psd_bm_truth, norm=norm_main, cmap=cmap))
plots_main.append(ax11.pcolormesh(111/psd_bm_QGSW.freq_r, 1/psd_bm_QGSW.freq_time, psd_bm_QGSW, norm=norm_main, cmap=cmap))
plots_main.append(ax20.pcolormesh(111/psd_it_truth.freq_r, 1/psd_it_truth.freq_time, psd_it_truth, norm=norm_main, cmap=cmap))
plots_main.append(ax21.pcolormesh(111/psd_it_QGSW.freq_r, 1/psd_it_QGSW.freq_time, psd_it_QGSW, norm=norm_main, cmap=cmap))

# Third column
plots_right = []
plots_right.append(ax02.pcolormesh(111/psd_tot_truth.freq_r, 1/psd_tot_truth.freq_time, 1-psd_err_tot/psd_tot_truth, vmin=0, vmax=1, cmap=cmap_score))
plots_right.append(ax12.pcolormesh(111/psd_bm_truth.freq_r, 1/psd_bm_truth.freq_time, 1-psd_err_bm/psd_bm_truth, vmin=0, vmax=1, cmap=cmap_score))
plots_right.append(ax22.pcolormesh(111/psd_it_truth.freq_r, 1/psd_it_truth.freq_time, 1-psd_err_it/psd_it_truth, vmin=0, vmax=1, cmap=cmap_score))


ax00.set_title(r"$\eta_{REF}^{TOT}$ reference field",fontsize=14)
ax01.set_title(r"$\eta^{TOT}$ reconstructed field",fontsize=14)
ax02.set_title(r"$\eta^{TOT}$ PSD score",fontsize=14)
ax10.set_title(r"$\eta_{REF}^{BM}$ reference field",fontsize=14)
ax11.set_title(r"$\eta^{BM}$ reconstructed field",fontsize=14)
ax12.set_title(r"$\eta^{BM}$ PSD score",fontsize=14)
ax20.set_title(r"$\eta_{REF}^{IT}$ reference field",fontsize=14)
ax21.set_title(r"$\eta^{IT}$ reconstructed field",fontsize=14)
ax22.set_title(r"$\eta^{IT}$ PSD score",fontsize=14)

# ----------------------------
# Set log scales, inverted axes, labels
all_axes = [ax00, ax01, ax02, ax10, ax11, ax12, ax20, ax21, ax22]
for i,_ax in enumerate(all_axes):
    _ax.set_xscale("log")
    _ax.set_yscale("log")
    _ax.xaxis.set_inverted(True)
    _ax.yaxis.set_inverted(True)
    _ax.set_ylim(30, 8/24)
    if i in [0,3,6]:
        _ax.set_ylabel("Period [days]", fontsize=12)
    if i in [6,7,8]:
        _ax.set_xlabel("Wavelength [km]", fontsize=12)

    xticks = [20,100, 300]

    # Set ticks
    _ax.set_xscale("log")
    _ax.set_xticks(xticks)

    # # Force labels exactly how you want
    _ax.set_xticklabels([ r"$2\times 10^1$",r"$10^2$", r"$3\times 10^2$"], fontsize=10, color="black")

    # Major ticks
    _ax.xaxis.set_major_locator(LogLocator(base=10))
    _ax.yaxis.set_major_locator(LogLocator(base=10))

    # Minor ticks
    _ax.xaxis.set_minor_locator(LogLocator(base=10, subs='auto'))
    _ax.yaxis.set_minor_locator(LogLocator(base=10, subs='auto'))

    # Grid
    _ax.grid(which='major', color='dimgrey', linestyle='-', linewidth=0.4, alpha=0.6)
    _ax.grid(which='minor', color='dimgrey', linestyle=':', linewidth=0.3, alpha=0.5)

# ----------------------------
# Colorbars using fig.add_axes for full control
# Position = [left, bottom, width, height]
cbar_ax_main = fig.add_axes([0.605, 0.2, 0.015, 0.6])  # between left and right sections
cbar_main = fig.colorbar(plots_main[0], cax=cbar_ax_main, extend='both', shrink=0.9, aspect=30)
cbar_main.ax.set_title("PSD", fontsize=12)

cbar_ax_right = fig.add_axes([0.92, 0.2, 0.015, 0.6])  # right of third column
cbar_right = fig.colorbar(plots_right[0], cax=cbar_ax_right, shrink=0.9, aspect=30)
cbar_right.ax.set_title("       PSD score", fontsize=12)

plt.show()


In [ ]:
t_M2 = 12.42060121
t_S2 = 12
t_N2 = 12.65834751

In [ ]:
c_text = 'black' # color of the texts 

def plot_transparent_zoom(psd,name,**kwargs):

    fig, ax = plt.subplots(1,1,figsize=(4,2),)

    pc = ax.pcolormesh(111/psd.freq_r,
                    (1/psd.freq_time)*24,
                    psd,
                    **kwargs)

    ax.axhline(t_M2,c=c_text,linestyle="dashed")
    ax.axhline(t_S2,c=c_text,linestyle="dashed")
    ax.axhline(t_N2,c=c_text,linestyle="dashed")

    ax.set_xscale("log")
    ax.xaxis.set_inverted(True)
    ax.yaxis.set_inverted(True)
    ax.set_ylim(13, 11)
    ax.set_xlim(300, 80)


    # Ticks + numbers
    ax.tick_params(colors=c_text)
    ax.tick_params(axis='both', which='major', labelsize=14, colors=c_text)
    ax.tick_params(axis='both', which='minor', labelsize=14, colors=c_text)

    # Frame
    for spine in ax.spines.values():
        spine.set_edgecolor(c_text)

    # Offset text
    ax.xaxis.get_offset_text().set_color(c_text)
    ax.yaxis.get_offset_text().set_color(c_text)

    # Tick labels (including log formatter ones)
    for label in ax.get_xticklabels(which="both") + ax.get_yticklabels(which="both"):
        label.set_color(c_text)

    # Axis labels
    ax.set_ylabel("Period [hours]", fontsize=16, color=c_text)
    ax.set_xlabel("Wavelength [km]", fontsize=16, color=c_text)

    plt.savefig(f"{name}.png", transparent=True, dpi=300, bbox_inches="tight", pad_inches=0.1)



In [ ]:
plot_transparent_zoom(psd_it_truth,"transparent_zoom_it_truth",norm=norm_main, cmap=cmap)
plot_transparent_zoom(psd_it_QGSW,"transparent_zoom_it_QGSW",norm=norm_main, cmap=cmap)
plot_transparent_zoom(1-psd_err_tot/psd_tot_truth,"transparent_zoom_it_PSD_score", vmin=0, vmax=1, cmap=cmap_score)

## Plot Figure **05**

In [ ]:
psd_err_tot_QG = _psd(ssh_bm_QG-ssh_tot_truth)

psd_err_tot_QG = psd_err_tot_QG.where(psd_err_tot_QG.freq_r>0.4,drop=True)

In [ ]:
cmap_score = cc.cm.CET_L18

fig = plt.figure(figsize=(8, 3.8), dpi=300)

gs = fig.add_gridspec(1, 2, wspace=0.2)

ax00 = fig.add_subplot(gs[0])
ax01 = fig.add_subplot(gs[1])

plots_main = []
plots_main.append(ax00.pcolormesh(111/psd_tot_truth.freq_r, 1/psd_tot_truth.freq_time, 1-psd_err_tot_QG/psd_tot_truth, vmin=0, vmax=1, cmap=cmap_score))
plots_main.append(ax01.pcolormesh(111/psd_tot_truth.freq_r, 1/psd_tot_truth.freq_time, 1-psd_err_tot/psd_tot_truth, vmin=0, vmax=1, cmap=cmap_score))

ax00.set_title(r"$QG$",fontsize=16)
ax01.set_title(r"$QG/SW$",fontsize=16)

all_axes = [ax00, ax01]
for i,_ax in enumerate(all_axes):
    _ax.set_xscale("log")
    _ax.set_yscale("log")
    _ax.xaxis.set_inverted(True)
    _ax.yaxis.set_inverted(True)
    _ax.set_ylim(30, 8/24)
    if i in [0]:
        _ax.set_ylabel("Period [days]", fontsize=12)
    _ax.set_xlabel("Wavelength [km]", fontsize=12)

    xticks = [20,100, 300]

    # Set ticks
    _ax.set_xscale("log")
    _ax.set_xticks(xticks)

    # # Force labels exactly how you want
    _ax.set_xticklabels([ r"$2\times 10^1$",r"$10^2$", r"$3\times 10^2$"], fontsize=10, color="black")

    # Major ticks
    _ax.xaxis.set_major_locator(LogLocator(base=10))
    _ax.yaxis.set_major_locator(LogLocator(base=10))

    # Minor ticks
    _ax.xaxis.set_minor_locator(LogLocator(base=10, subs='auto'))
    _ax.yaxis.set_minor_locator(LogLocator(base=10, subs='auto'))

    # Grid
    _ax.grid(which='major', color='dimgrey', linestyle='-', linewidth=0.4, alpha=0.6)
    _ax.grid(which='minor', color='dimgrey', linestyle=':', linewidth=0.3, alpha=0.5)

cbar_ax_right = fig.add_axes([0.94, 0.12, 0.015, 0.7])  # right of third column
cbar_right = fig.colorbar(plots_right[0], cax=cbar_ax_right, shrink=0.9, aspect=30)
cbar_right.ax.set_title("       PSD score", fontsize=12)

plt.show()

## Plot Figure **07**

In [ ]:
def _psd(da,dim=None,dim_iso=None,dim_mean=None,detrend='constant'):

    # Rechunk 
    if dim is not None:
        chunks = {}
        if type(dim)!=list:
            dim = [dim]
        for d in dim:
            chunks[d] = da[d].size
        da = da.chunk(chunks)

    # Compute PSD
    psd = xrft.power_spectrum(
        da, 
        dim=dim, 
        detrend=detrend, 
        window=True).compute()
    
    # Isotropize
    if dim_iso is not None:
        psd = xrft.isotropize(psd,fftdim=[f'freq_{dim_iso[0]}',f'freq_{dim_iso[1]}'])

    # Positive coordinates
    ispos = True
    for d in psd.dims:
        ispos = ispos & (psd[d] > 0.)
    psd = psd.where(ispos, drop=True)
    
    # Average
    if dim_mean is not None:
        psd = psd.mean(dim=dim_mean)
    
    psd.name = f'PSD_{da.name}'
    
    return psd

In [ ]:
psd_bm_truth = _psd(ssh_bm_truth,dim_iso=["longitude","latitude"],dim_mean='freq_time')

In [ ]:
psd_err_bm_QG = _psd(ssh_bm_QG-ssh_bm_truth,dim_iso=["longitude","latitude"],dim_mean='freq_time')

In [ ]:
psd_err_bm_QGSW = _psd(ssh_bm_QGSW-ssh_bm_truth,dim_iso=["longitude","latitude"],dim_mean='freq_time')

In [ ]:
psd_err_bm_QGSW_notime = _psd(ssh_bm_QGSW_notime-ssh_bm_truth,dim_iso=["longitude","latitude"],dim_mean='freq_time')

In [ ]:
# plt.plot(111*1/psd_bm_truth.freq_r,psd_bm_truth,c='black')

plt.figure(figsize=(8,4.5),dpi=300)

c_QG = 'forestgreen'
c_QGSW = 'darkblue'
c_QGSW_N = 'firebrick'

# PSD SCORE LINES #
plt.plot(111*1/psd_err_bm_QG.freq_r,1-psd_err_bm_QG/psd_bm_truth,c=c_QG,linewidth=3,label="QG")
plt.plot(111*1/psd_err_bm_QGSW.freq_r,1-psd_err_bm_QGSW/psd_bm_truth,c=c_QGSW,linewidth=3,label="QG/SW")
plt.plot(111*1/psd_err_bm_QGSW_notime.freq_r,1-psd_err_bm_QGSW_notime/psd_bm_truth,c=c_QGSW_N,linewidth=3,label="QG/SW N")

# O.5 limit #
plt.axhline(0.5,linestyle='dashed',c='black',linewidth=1.5)

# EFFECTIVE RESOLUTION LINES #
plt.plot([162.9945198720743,162.9945198720743],[0,0.5],c=c_QG,linestyle='dashed',linewidth=1.5)
plt.plot([116.99403328908684,116.99403328908684],[0,0.5],c=c_QGSW,linestyle='dashed',linewidth=1.5)
plt.plot([131.9482023355848,131.9482023355848],[0,0.5],c=c_QGSW_N,linestyle='dashed',linewidth=1.5)


plt.xlabel("Wavelength [km]",fontsize=16)
plt.ylabel("PSD Score",fontsize=16)

plt.grid(True, which="both", linestyle="--", linewidth=0.5)

plt.legend(fontsize=14)

plt.ylim(0,1)
plt.xlim(60,400)
plt.xscale('log')
# plt.yscale('log')
plt.gca().invert_xaxis()
